# 📊 Enriched Dataset EDA & Visualizations

This notebook provides a visual breakdown of the three datasets in the Hybrid Recommender project:
1. **Industrial & Scientific**
2. **Video Games**
3. **Cell Phones & Accessories**

We focus on **Ratings Class Balance** and **User Interaction History Density**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
def load_and_prep(file_path, name):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None
    
    print(f"Loading {name}...")
    # Load only necessary columns for speed
    df = pd.read_csv(file_path, usecols=['rating', 'history'])
    df['seq_len'] = df['history'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)
    return df

datasets = {
    "Industrial & Scientific": "../data/train_industrial_and_scientific_merged.csv",
    "Video Games": "../data/train_video_games_merged.csv",
    "Cell Phones": "../data/train_cell_phones_and_accessories_merged.csv"
}

data_frames = {}
for name, path in datasets.items():
    df = load_and_prep(path, name)
    if df is not None:
        data_frames[name] = df

## 1. Ratings Class Balance
How biased are our reviews? (Spoiler: Very biased toward 5 stars)

In [ ]:
if data_frames:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for i, (name, df) in enumerate(data_frames.items()):
        counts = df['rating'].value_counts().sort_index()
        sns.barplot(x=counts.index, y=counts.values, ax=axes[i], palette="viridis", hue=counts.index, legend=False)
        axes[i].set_title(f"Ratings: {name}")
        axes[i].set_ylabel("Count")
        axes[i].set_xlabel("Star Rating")

    plt.tight_layout()
    plt.show()
else:
    print("No data frames loaded. Check your data folder paths.")

## 2. Interaction History Density
Distribution of sequence lengths for SASRec. This shows how much context each user provides.

In [ ]:
if data_frames:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for i, (name, df) in enumerate(data_frames.items()):
        # Filter outliers for better visualization (95th percentile)
        threshold = df['seq_len'].quantile(0.95)
        sns.histplot(df[df['seq_len'] <= threshold]['seq_len'], bins=20, ax=axes[i], kde=True, color="teal")
        axes[i].set_title(f"Seq Lengths: {name} (Top 95%)")
        axes[i].set_xlabel("Number of Past Interactions")

    plt.tight_layout()
    plt.show()

## 3. Summary Statistics Table

In [ ]:
summary = []
for name, df in data_frames.items():
    summary.append({
        "Dataset": name,
        "Total Interactions": len(df),
        "Avg Rating": round(float(df['rating'].mean()), 2),
        "Median Seq Len": df['seq_len'].median(),
        "Max Seq Len": df['seq_len'].max(),
        "5-Star %": round(float(len(df[df['rating'] == 5.0]) / len(df) * 100), 2)
    })

pd.DataFrame(summary)